[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week06.ipynb)

# Week 6: Optimization
**Introduction to Deep Learning (HIT)** &middot; Part II: Training Infrastructure

Gradient descent; SGD, momentum, and Adam; learning rates and optimization dynamics.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Understand SGD, momentum, and Adam.
- Reason about learning rates and convergence.
- Tune an optimizer to a target.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. One model, three optimizers
SGD, SGD+momentum, and Adam on the same problem.

In [ ]:
torch.manual_seed(0)
X = torch.randn(200, 5); y = X @ torch.randn(5, 1) + 0.1 * torch.randn(200, 1)

def run(name, steps=80):
    torch.manual_seed(0); m = nn.Linear(5, 1); f = nn.MSELoss()
    o = {"SGD": torch.optim.SGD(m.parameters(), lr=0.05),
         "Momentum": torch.optim.SGD(m.parameters(), lr=0.05, momentum=0.9),
         "Adam": torch.optim.Adam(m.parameters(), lr=0.05)}[name]
    h = []
    for _ in range(steps):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    return h

for name in ["SGD", "Momentum", "Adam"]:
    plt.plot(run(name), label=name)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Optimizer comparison"); plt.show()

## 2. The learning rate is everything
A three-rate sweep: too small, good, too large.

In [ ]:
for lr in [0.001, 0.05, 0.5]:
    torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=lr); f = nn.MSELoss()
    h = []
    for _ in range(80):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label=f"lr={lr}")
plt.yscale("log"); plt.legend(); plt.xlabel("step"); plt.ylabel("loss (log)"); plt.show()

## 3. Read the curve to diagnose
Spikes = too large; flat = too small; smooth decline = good.

In [ ]:
# deliberately too large -> watch it blow up
torch.manual_seed(0); m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=1.2); f = nn.MSELoss()
h = []
for _ in range(40):
    o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
print("loss exploded to:", h[-1])
plt.plot(h); plt.title("lr too large: divergence"); plt.xlabel("step"); plt.ylabel("loss"); plt.show()

## 4. A learning-rate schedule
Step decay: lower the rate as training settles.

In [ ]:
m = nn.Linear(5, 1); o = torch.optim.SGD(m.parameters(), lr=0.2); f = nn.MSELoss()
sched = torch.optim.lr_scheduler.StepLR(o, step_size=25, gamma=0.3)
lrs, h = [], []
for _ in range(80):
    o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); sched.step()
    lrs.append(o.param_groups[0]["lr"]); h.append(l.item())
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(lrs); ax[0].set_title("learning rate"); ax[1].plot(h); ax[1].set_title("loss")
plt.show()

**Try it live:** swap `StepLR` for `CosineAnnealingLR(o, T_max=80)` and compare the curves.

---
Student materials for this week: the lab handout (`labs/week06.html`) and the curated references (`references/week06.html`) in the course site.